<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

Prepended http:// to 'songs_dataset.zip'
--2026-09-09 17:54:24--  http://songs_dataset.zip/
Resolving songs_dataset.zip (songs_dataset.zip)... failed: nodename nor servname provided, or not known.
wget: unable to resolve host address ‘songs_dataset.zip’
--2026-09-09 17:54:24--  https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip [following]
--2026-09-09 17:54:26--  https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154

In [3]:
# Posibles bandas
os.listdir("./songs_dataset/")

['prince.txt',
 'dickinson.txt',
 'notorious-big.txt',
 'beatles.txt',
 'bob-dylan.txt',
 'bjork.txt',
 'johnny-cash.txt',
 'disney.txt',
 'janisjoplin.txt',
 'kanye.txt',
 'bob-marley.txt',
 'leonard-cohen.txt',
 'ludacris.txt',
 'adele.txt',
 'alicia-keys.txt',
 'joni-mitchell.txt',
 'amy-winehouse.txt',
 'lorde.txt',
 'rihanna.txt',
 'Kanye_West.txt',
 'nirvana.txt',
 'cake.txt',
 'bieber.txt',
 'notorious_big.txt',
 'missy-elliott.txt',
 'dolly-parton.txt',
 'jimi-hendrix.txt',
 'michael-jackson.txt',
 'al-green.txt',
 'lil-wayne.txt',
 'lady-gaga.txt',
 'lin-manuel-miranda.txt',
 'nursery_rhymes.txt',
 'dj-khaled.txt',
 'radiohead.txt',
 'patti-smith.txt',
 'blink-182.txt',
 'Lil_Wayne.txt',
 'dr-seuss.txt',
 'r-kelly.txt',
 'drake.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'nicki-minaj.txt',
 'kanye-west.txt',
 'paul-simon.txt',
 'nickelback.txt',
 'eminem.txt',
 'bruno-mars.txt']

In [4]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

/var/folders/6k/5h802l7x7nl4z6q7y2xjf4qc0000gn/T/ipykernel_68378/3849064916.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [6]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [7]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [8]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [9]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [10]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [11]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [12]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


### 3 - Entrenar embeddings

In [13]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 113045.2734375
Loss after epoch 1: 65966.6171875
Loss after epoch 2: 65934.984375
Loss after epoch 3: 65718.4375
Loss after epoch 4: 63875.0
Loss after epoch 5: 64160.75
Loss after epoch 6: 64080.28125
Loss after epoch 7: 64814.84375
Loss after epoch 8: 62632.75
Loss after epoch 9: 60452.5625
Loss after epoch 10: 59839.9375
Loss after epoch 11: 58883.875
Loss after epoch 12: 57715.9375
Loss after epoch 13: 56494.125
Loss after epoch 14: 55817.5
Loss after epoch 15: 55843.0625
Loss after epoch 16: 51722.4375
Loss after epoch 17: 49858.25
Loss after epoch 18: 49592.125
Loss after epoch 19: 48960.0


(156986, 287740)

### 4 - Ensayar

In [14]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('pretty', 0.895426332950592),
 ('sleep', 0.866565465927124),
 ('help', 0.8439399600028992),
 ('cry', 0.8351276516914368),
 ('not', 0.8309584259986877),
 ('try', 0.8276942372322083),
 ('peace', 0.8144830465316772),
 ('little', 0.8140553832054138),
 ('twist', 0.8123889565467834),
 ('seems', 0.8079559803009033)]

In [15]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('shake', -0.22873042523860931),
 ('four', -0.233018159866333),
 ('five', -0.23746557533740997),
 ('six', -0.2378406822681427),
 ('bang', -0.2483215481042862),
 ('our', -0.25538885593414307),
 ('day', -0.2689799666404724),
 ('going', -0.2692091464996338),
 ('here', -0.26990947127342224),
 ('three', -0.28389406204223633)]

In [16]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('five', 0.9813726544380188),
 ('three', 0.9745770692825317),
 ('six', 0.9710814952850342),
 ('seven', 0.9584383964538574),
 ('two', 0.9517245292663574),
 ('sixty', 0.899040937423706),
 ('one', 0.7951183319091797),
 ('crying', 0.7946316003799438),
 ('us', 0.7740064263343811),
 ("i'm", 0.75083988904953)]

In [17]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[("can't", 0.9434003233909607),
 ('buy', 0.9396981000900269),
 ('much', 0.903316080570221),
 ('just', 0.8509079813957214),
 ('hide', 0.8355334997177124)]

In [18]:
# Ensayar con una palabra que no está en el vocabulario:
w2v_model.wv.most_similar(negative=["diedaa"])

KeyError: "Key 'diedaa' not present in vocabulary"

In [19]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[ 0.06138991  0.05881394 -0.06370379  0.02445008 -0.2015189  -0.18612279
 -0.15284604  0.4548658  -0.04218334  0.03535625  0.13658    -0.1851993
 -0.18126647  0.22149746 -0.30380708 -0.23970437  0.0709502  -0.05679061
 -0.05166298 -0.23843724 -0.08529764  0.19564407 -0.0767937   0.03796828
  0.07515893 -0.04826084  0.07379368  0.10396545  0.00737698 -0.22764525
 -0.04566917  0.12937078  0.27785707  0.1938714  -0.1350965   0.2085755
  0.4091737  -0.00386647 -0.10631301 -0.09056874  0.02401084 -0.08004661
  0.13398568  0.08833543 -0.01894663  0.08592883 -0.15905924  0.10259863
  0.1445956  -0.12092476 -0.27919465 -0.04061609  0.11381607  0.31366462
 -0.07408216  0.13977578  0.22791366  0.1320914  -0.01811455  0.09772391
  0.0925008  -0.1487154  -0.16348435 -0.13202292 -0.09834062  0.02714083
  0.1653129   0.26052263 -0.03259885 -0.02894831  0.11621293 -0.06974422
  0.09563003 -0.1527615   0.2207087   0.15996675  0.1589054  -0.04711805
 -0.12555768 -0.03993415 -0.10794961  0.01879169  0.0

In [20]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 1.0),
 ('babe', 0.9085179567337036),
 ('someone', 0.8886149525642395),
 ('need', 0.8828009963035583),
 ('nothing', 0.8740281462669373),
 ("didn't", 0.8638388514518738),
 ("there's", 0.8526664972305298),
 ('you', 0.8456774950027466),
 ('feed', 0.8445045351982117),
 ('somebody', 0.8362816572189331)]

In [21]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.9085179567337036),
 ('someone', 0.8886149525642395),
 ('need', 0.8828009963035583),
 ('nothing', 0.8740281462669373),
 ("didn't", 0.8638389110565186),
 ("there's", 0.8526665568351746),
 ('you', 0.8456774950027466),
 ('feed', 0.8445045351982117),
 ('somebody', 0.8362816572189331),
 ('buy', 0.8351728916168213)]

### 5 - Visualizar agrupación de vectores

In [22]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [23]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show()

In [24]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show()

In [25]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

## Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

---

**1. Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio.**

Como corpus elegí las letras de Las Pastillas del Abuelo, una banda de rock argentino que escucho bastante. Me pareció un buen candidato porque tiene un vocabulario argentino muy marcado. Eso me da términos de interés claros para buscar vecinos y, si el corpus alcanza, debería mostrar grupos interpretables en la visualización.

Las letras las bajé de letras.com (https://www.letras.com/las-pastillas-del-abuelo/) con el script `scrap_letras.py` que está en esta misma carpeta. El script recorre la lista de canciones del artista, descarta las versiones repetidas (acústicos, en vivo, colaboraciones) quedándose con una sola letra por canción, y guarda un verso por línea en `dataset/pastillas_del_abuelo.txt`.

#### Carga del corpus

In [26]:
from pathlib import Path

versos = Path("dataset/pastillas_del_abuelo.txt").read_text(encoding="utf-8").splitlines()
canciones = pd.read_csv("dataset/canciones.csv")
display(canciones.head(10))

,titulo,url,versos
0,Tarot,https://www.letras.com/las-pastillas-del-abuel...,49
1,La Doctora II,https://www.letras.com/las-pastillas-del-abuel...,66
2,Viejo Karma!,https://www.letras.com/las-pastillas-del-abuel...,57
3,¿Qué Es Dios?,https://www.letras.com/las-pastillas-del-abuel...,59
4,Hasta Acá Nos Ayudó Dios!,https://www.letras.com/las-pastillas-del-abuel...,48
5,Donde Esconder Tantas Manos?,https://www.letras.com/las-pastillas-del-abuel...,56
6,Otra Vuelta de Tuerca,https://www.letras.com/las-pastillas-del-abuel...,80
7,Princesa,https://www.letras.com/las-pastillas-del-abuel...,34
8,Ojos de Dragón!,https://www.letras.com/las-pastillas-del-abuel...,50
9,El Sensei,https://www.letras.com/las-pastillas-del-abuel...,112


In [27]:
# el txt tiene los versos en el mismo orden que el csv, asi que con los conteos
# reconstruyo cada cancion como una lista de versos
letras, i = [], 0
for n in canciones["versos"]:
    letras.append(versos[i:i + n])
    i += n

print(f"Cantidad de canciones: {len(letras)}")
print(f"Cantidad de versos: {len(versos)}")
print(f"Versos por cancion en promedio: {len(versos) / len(letras):.1f}")
print("\n".join(letras[0][:6]))

Cantidad de canciones: 141
Cantidad de versos: 5147
Versos por cancion en promedio: 36.5
Puede que seas
La emperatriz que rebardea
A la papisa que la frena
Con fundamentos de esquimal
Y el mundo crea
Que sos la estrella que se estrella


Cada canción va a ser un documento. Probé también usar cada verso como documento, igual que el ejemplo de los Beatles, pero los versos tienen unas 6 palabras y con eso el modelo solo aprende qué palabras estan en el mismo verso. Con la canción entera sacamos los saltos de linea y las palabras tienen mas contexto.

### Preprocesamiento

In [28]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

# a los filtros default de keras le sumo los signos de apertura y comillas del espanol
FILTROS = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n' + '¿¡«»…'

# un documento por cancion, con los tokens de todos sus versos seguidos
oraciones = []
for letra in letras:
    tokens = []
    for verso in letra:
        tokens += text_to_word_sequence(verso, filters=FILTROS)
    oraciones.append(tokens)

print(f"Documentos: {len(oraciones)}")
print(f"Tokens totales: {sum(len(o) for o in oraciones)}")
print(f"Tokens por documento en promedio: {sum(len(o) for o in oraciones) / len(oraciones):.0f}")
oraciones[0][:20]

Documentos: 141
Tokens totales: 30168
Tokens por documento en promedio: 214


['puede',
 'que',
 'seas',
 'la',
 'emperatriz',
 'que',
 'rebardea',
 'a',
 'la',
 'papisa',
 'que',
 'la',
 'frena',
 'con',
 'fundamentos',
 'de',
 'esquimal',
 'y',
 'el',
 'mundo']

In [29]:
from collections import Counter

frecuencia = Counter(t for o in oraciones for t in o)

print(f"Palabras distintas: {len(frecuencia)}")
for umbral in (2, 3, 5):
    print(f"Con frecuencia >= {umbral}: {sum(f >= umbral for f in frecuencia.values())}")

pd.DataFrame(frecuencia.most_common(20), columns=["palabra", "frecuencia"])

Palabras distintas: 5124
Con frecuencia >= 2: 2437
Con frecuencia >= 3: 1435
Con frecuencia >= 5: 778


,palabra,frecuencia
0,que,1342
1,de,1024
2,y,974
3,no,822
4,el,819
5,la,817
6,a,670
7,en,591
8,un,488
9,me,349


Las palabras más frecuentes son stopwords como "que", "de" e "y". Las dejo porque word2vec las usa como contexto y el corpus es chico. Si hay que tener en cuenta al momento de entrenar que mas de la mitad de las palabras aparecen solo una vez.

### Entrenamiento
Decisiones:
* Uso skip-gram con 20 ejemplos negativos como en el ejemplo de clase.
* Bajo min_count=3, porque como se ve en la tabla anterior con frecuencia 5 quedan pocas palabras para entrenar. 
* Uso window=4 que me funciono bien experimentalmente
* vector_size=70 porque probe con 300 tambien y me dio resultados similares, entonces considero que es una dimension suficiente para la cantidad de datos. 
* 100 épocas porque con pocos documentos cada pasada aporta poco.
* seed=42 y un solo worker para que el resultado sea reproducible.

In [30]:
w2v_model = Word2Vec(min_count=3,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=4,       # cant de palabras antes y desp de la predicha
                     vector_size=70, # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1,           # modelo 0:CBOW  1:skipgram
                     seed=42)        # semilla para reproducibilidad

In [31]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [32]:
print("Cantidad de docs:", w2v_model.corpus_count)

Cantidad de docs: 1846


In [33]:
print("Tamano del vocabulario:", len(w2v_model.wv.index_to_key))

Tamano del vocabulario: 637


In [34]:
# reuso la clase callback definida arriba para ver el loss por epoca
w2v_model.train(oraciones,
                total_examples=w2v_model.corpus_count,
                epochs=100,
                compute_loss=True,
                callbacks=[callback()])

Loss after epoch 0: 21384.4375
Loss after epoch 1: 7159.517578125
Loss after epoch 2: 6562.001953125
Loss after epoch 3: 6245.4140625
Loss after epoch 4: 6210.609375
Loss after epoch 5: 6274.87109375
Loss after epoch 6: 6193.72265625
Loss after epoch 7: 6416.68359375
Loss after epoch 8: 6286.0234375
Loss after epoch 9: 6565.7109375
Loss after epoch 10: 6156.8515625
Loss after epoch 11: 6169.109375
Loss after epoch 12: 6361.3046875
Loss after epoch 13: 6487.484375
Loss after epoch 14: 6267.6953125
Loss after epoch 15: 6213.890625
Loss after epoch 16: 6133.4375
Loss after epoch 17: 6263.5390625
Loss after epoch 18: 6530.6796875
Loss after epoch 19: 6312.703125
Loss after epoch 20: 6271.171875
Loss after epoch 21: 6465.484375
Loss after epoch 22: 6144.640625
Loss after epoch 23: 6102.140625
Loss after epoch 24: 6382.453125
Loss after epoch 25: 6024.265625
Loss after epoch 26: 6308.484375
Loss after epoch 27: 6033.625
Loss after epoch 28: 6241.484375
Loss after epoch 29: 6439.90625
Loss af

(198511, 3016800)